In [1]:
%pip install -q --upgrade langchain 
%pip install -q --upgrade pypdf
%pip install -q --upgrade faiss-gpu
%pip install -q --upgrade langchain-community
%pip install -q --upgrade langchain-aws
%pip install -q --upgrade faiss-cpu
%pip install -q --upgrade boto3

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Langchain com AWS Bedrock para RAG (Retrieval Augmented Generation)
# Este script lê PDFs de uma pasta, cria embeddings usando AWS Bedrock,
# armazena-os em um FAISS VectorStore local e usa um LLM da AWS Bedrock
# para responder perguntas baseadas nos documentos.

# 1. Instalação de bibliotecas necessárias:
# pip install langchain langchain-community langchain-aws faiss-cpu pypdf boto3

import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_aws import BedrockEmbeddings # Atualizado para langchain_aws
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockLLM # Atualizado para langchain_aws
from langchain.chains import RetrievalQA



In [3]:
# --- Configurações ---
# Substitua pelos seus valores ou configure variáveis de ambiente
AWS_REGION = "us-east-1"  # Ex: "us-east-1", "us-west-2", etc.
# Modelos de Embeddings Populares na AWS Bedrock:
# - amazon.titan-embed-text-v1
# - cohere.embed-english-v3 / cohere.embed-multilingual-v3
BEDROCK_EMBEDDINGS_MODEL_ID = "amazon.titan-embed-text-v2:0"
# Modelos LLM Populares na AWS Bedrock:
# - anthropic.claude-v2:1
# - anthropic.claude-3-sonnet-20240229-v1:0
# - meta.llama2-70b-chat-v1
# - ai21.j2-ultra-v1
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"

PDF_DIRECTORY_PATH = "./documentos_pdf"  # Crie uma pasta chamada "documentos_pdf" e coloque seus PDFs nela
FAISS_INDEX_PATH = "meu_faiss_index"

# --- Funções Auxiliares ---

def carregar_documentos(caminho_diretorio: str):
    """
    Carrega documentos PDF de um diretório especificado.
    """
    print(f"Carregando documentos de: {caminho_diretorio}")
    if not os.path.exists(caminho_diretorio):
        print(f"ERRO: O diretório '{caminho_diretorio}' não foi encontrado. Crie-o e adicione arquivos PDF.")
        return None
    
    # Usando DirectoryLoader com PyPDFLoader para carregar todos os PDFs na pasta
    loader = DirectoryLoader(
        caminho_diretorio,
        glob="**/*.pdf", # Padrão para encontrar arquivos PDF
        loader_cls=PyPDFLoader,
        show_progress=True,
        use_multithreading=True # Pode acelerar o carregamento de múltiplos arquivos
    )
    documentos = loader.load()
    if not documentos:
        print("Nenhum documento PDF encontrado no diretório.")
        return None
    print(f"Total de documentos carregados: {len(documentos)}")
    return documentos

def dividir_documentos_em_chunks(documentos: list):
    """
    Divide os documentos carregados em chunks menores.
    """
    print("Dividindo documentos em chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,  # Tamanho de cada chunk
        chunk_overlap=200, # Sobreposição entre chunks para manter o contexto
        length_function=len
    )
    chunks = text_splitter.split_documents(documentos)
    print(f"Total de chunks criados: {len(chunks)}")
    return chunks

def inicializar_bedrock_embeddings(regiao: str, model_id: str):
    """
    Inicializa o cliente de embeddings do Bedrock.
    Certifique-se de que suas credenciais AWS estão configuradas
    (ex: via AWS CLI 'aws configure', variáveis de ambiente, ou roles IAM).
    """
    print(f"Inicializando Bedrock Embeddings com o modelo: {model_id} na região {regiao}")
    embeddings = BedrockEmbeddings(
        region_name=regiao,
        model_id=model_id
    )
    return embeddings

def criar_ou_carregar_vectorstore_faiss(chunks: list, embeddings_model, caminho_indice: str):
    """
    Cria um novo FAISS VectorStore se não existir, ou carrega um existente.
    """
    if os.path.exists(caminho_indice):
        print(f"Carregando VectorStore FAISS existente de: {caminho_indice}")
        vectorstore = FAISS.load_local(caminho_indice, embeddings_model, allow_dangerous_deserialization=True)
        print("VectorStore carregado com sucesso.")
    else:
        if not chunks:
            print("Nenhum chunk de documento para criar o VectorStore. Verifique o carregamento dos PDFs.")
            return None
        print(f"Criando novo VectorStore FAISS e salvando em: {caminho_indice}")
        vectorstore = FAISS.from_documents(chunks, embeddings_model)
        vectorstore.save_local(caminho_indice)
        print("Novo VectorStore criado e salvo com sucesso.")
    return vectorstore

def inicializar_bedrock_llm(regiao: str, model_id: str):
    """
    Inicializa o cliente LLM do Bedrock.
    """
    print(f"Inicializando Bedrock LLM com o modelo: {model_id} na região {regiao}")
    llm = BedrockLLM(
        region_name=regiao,
        model_id=model_id,
        model_kwargs={ # Argumentos específicos do modelo, ajuste conforme necessário
            "max_tokens_to_sample": 2000, # Para Claude
            "temperature": 0.1,
            # "top_p": 0.9 # Exemplo para outros modelos
        }
    )
    return llm

def criar_cadeia_qa(llm, vectorstore):
    """
    Cria a cadeia de RetrievalQA.
    """
    print("Criando a cadeia de RetrievalQA...")
    retriever = vectorstore.as_retriever(
        search_type="similarity", # Tipos: "similarity", "mmr", "similarity_score_threshold"
        search_kwargs={"k": 5} # Número de documentos relevantes a serem recuperados
    )
    
    # Configuração do prompt (opcional, Langchain tem um padrão)
    # from langchain.prompts import PromptTemplate
    # prompt_template = """Use o seguinte contexto para responder à pergunta no final.
    # Se você não sabe a resposta, apenas diga que não sabe, não tente inventar uma resposta.
    # Mantenha a resposta o mais concisa possível.
    # Contexto: {context}
    # Pergunta: {question}
    # Resposta útil:"""
    # PROMPT = PromptTemplate(
    #     template=prompt_template, input_variables=["context", "question"]
    # )
    # chain_type_kwargs = {"prompt": PROMPT}

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff", # Tipos: "stuff", "map_reduce", "refine", "map_rerank"
        retriever=retriever,
        return_source_documents=True, # Para ver quais documentos foram usados
        # chain_type_kwargs=chain_type_kwargs # Para usar o prompt customizado
    )
    print("Cadeia de QA criada com sucesso.")
    return qa_chain

# --- Fluxo Principal ---
if __name__ == "__main__":
    print("Iniciando processo de RAG com AWS Bedrock e FAISS...")

    

Iniciando processo de RAG com AWS Bedrock e FAISS...


In [4]:
# 1. Criar o diretório de PDFs se não existir (apenas para facilitar o primeiro uso)
if not os.path.exists(PDF_DIRECTORY_PATH):
    os.makedirs(PDF_DIRECTORY_PATH)
    print(f"Diretório '{PDF_DIRECTORY_PATH}' criado. Por favor, adicione seus arquivos PDF nele e rode o script novamente.")
    exit()

# 2. Carregar documentos PDF
documentos = carregar_documentos(PDF_DIRECTORY_PATH)



Carregando documentos de: ./documentos_pdf


100%|██████████| 1/1 [00:01<00:00,  1.94s/it]

Total de documentos carregados: 23


In [ ]:
if documentos:
    # 3. Dividir documentos em chunks
    chunks_de_texto = dividir_documentos_em_chunks(documentos)

    # 4. Inicializar embeddings do Bedrock
    bedrock_embeddings = inicializar_bedrock_embeddings(AWS_REGION, BEDROCK_EMBEDDINGS_MODEL_ID)

    # 5. Criar ou carregar VectorStore FAISS
    vector_store = criar_ou_carregar_vectorstore_faiss(chunks_de_texto, bedrock_embeddings, FAISS_INDEX_PATH)

    if vector_store:
        # 6. Inicializar LLM do Bedrock
        bedrock_llm = inicializar_bedrock_llm(AWS_REGION, BEDROCK_LLM_MODEL_ID)

        # 7. Criar cadeia de  QA
        qa_chain = criar_cadeia_qa(bedrock_llm, vector_store)

        # 8. Loop de Perguntas e Respostas
        print("\n--- Sistema de Perguntas e Respostas ---")
        print("Digite 'sair' para terminar.")
        while True:
            pergunta_usuario = input("\nSua pergunta: ")
            if pergunta_usuario.lower() == 'sair':
                break
            if not pergunta_usuario.strip():
                print("Por favor, digite uma pergunta.")
                continue

            print("Processando sua pergunta...")
            try:
                resultado = qa_chain.invoke({"query": pergunta_usuario}) # .invoke para Langchain >=0.1.0
                
                print("\nResposta:")
                print(resultado["result"])

                # Opcional: mostrar documentos fonte
                print("\nDocumentos fonte recuperados:")
                for i, doc in enumerate(resultado["source_documents"]):
                    print(f"  Fonte {i+1}: Trecho de '{doc.metadata.get('source', 'N/A')}' (Página: {doc.metadata.get('page', 'N/A')})")
                    # print(f"    Conteúdo: {doc.page_content[:200]}...") # Descomente para ver o início do conteúdo
            except Exception as e:
                print(f"Ocorreu um erro ao processar a pergunta: {e}")
    else:
        print("Não foi possível criar ou carregar o VectorStore. Encerrando.")
else:
    print("Nenhum documento foi carregado. Encerrando.")

print("\nProcesso finalizado.")



Dividindo documentos em chunks...
Total de chunks criados: 34
Inicializando Bedrock Embeddings com o modelo: amazon.titan-embed-text-v2:0 na região us-east-1
Carregando VectorStore FAISS existente de: meu_faiss_index
VectorStore carregado com sucesso.
Inicializando Bedrock LLM com o modelo: amazon.nova-lite-v1:0 na região us-east-1
Criando a cadeia de RetrievalQA...
Cadeia de QA criada com sucesso.

--- Sistema de Perguntas e Respostas ---
Digite 'sair' para terminar.
Processando sua pergunta...


Error raised by bedrock service
Traceback (most recent call last):
  File "c:\Users\yagor\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_aws\llms\bedrock.py", line 935, in _prepare_input_and_invoke
    response = self.client.invoke_model(**request_options)
  File "c:\Users\yagor\AppData\Local\Programs\Python\Python313\Lib\site-packages\botocore\client.py", line 595, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\yagor\AppData\Local\Programs\Python\Python313\Lib\site-packages\botocore\context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\yagor\AppData\Local\Programs\Python\Python313\Lib\site-packages\botocore\client.py", line 1058, in _make_api_call
    raise error_class(parsed_response, operation_name)
botocore.errorfactory.ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: Malformed input reque

Ocorreu um erro ao processar a pergunta: An error occurred (ValidationException) when calling the InvokeModel operation: Malformed input request: #: required key [messages] not found, please reformat your input and try again.
Por favor, digite uma pergunta.
Por favor, digite uma pergunta.

Processo finalizado.
